# Модель поведенческого скринга для прогноза просрочек по кредитам 

---
# Описание

### Общая информация
- заказчик проекта: крупный розничный банк
- Массовые просрочки по кредитным платежам могут запустить цепную реакцию: рост резервов -> снижение ликвидности -> риск закрытия банка.

### Бизнес задача
- разработать инструмент прогноза просрочек платежей, опираясь на текущее поведение клиентов. Точный прогноз просрочек платежа позволяет управлять ликвидностью, то есть сглаживать резкие колебания.

### Цель
- разработать модель поведенческого скоринга, которая по данным о клиенте в выбранный месяц предсказывает вероятность того, что клиент банка совершит просрочку платежа по кредиту длительностью от 90 дней.

### ML задача

Вид задачи:
- тип: обучение с учителем
- подтип: классификация

Целевая переменная:
- целевая переменная — столбец, содержащий данные о том, будет ли у клиента просрочка от 90 дней в течение ближайших 12 месяцев
- бинарный признак: 1 - просрочка была, 0 - просрочки не было. Строится по принципу скользящего окна

Особенности:
- необходимо использовать горизонт прогноза 12 месяцев
- Объект моделирования — это клиент в конкретный месяц
- У одного и того же клиента значения целевой переменной могут различаться от месяца к месяцу. Даже если клиент ранее уже был в дефолте, в следующие месяцы он мог выплатить долг и вернуться к значению 0 целевой переменной

Ключевые метрики:
- Approval rate, «уровень одобрения» = $\frac{FN + TN}{N}$
- Default rate, «доля просрочек платежа» = $\frac{FN}{FN + TN}$
- Missed defaults rate, «доля пропущенных дефолтов» = $\frac{FN}{FN + TP} = 1 - Recall$ 


Требования по качеству модели:
- Approval rate не менее 65%
- Default rate не более 2%
- Missed defaults rate не более 4%

---
## 1. Настройка проекта и загрузка данных

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
import requests
import warnings
from phik import phik_matrix
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error
from typing import Optional, Union, Dict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_validate, cross_val_score
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
import optuna 
from datetime import datetime, timedelta
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from mlxtend.evaluate.time_series import (
    GroupTimeSeriesSplit,
    plot_splits,
    print_cv_info,
    print_split_info,
) 
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# воспроизводимость  вычислений
RANDOM_SEED = 42 

In [ ]:
# настройки визуалиции 
plt.rcParams["figure.figsize"] = (12, 8)

### Загрузка данных

In [ ]:
# пути к файлам
path_s: list[str] = [
    'https://code.s3.yandex.net/datasets/ds_15_loan_payment_credit.csv',
    'https://code.s3.yandex.net/datasets/ds_15_transactions.csv',
    'https://code.s3.yandex.net/datasets/ds_15_client_description.csv',
    'https://code.s3.yandex.net/datasets/ds_15_credit_description.csv',
    'https://code.s3.yandex.net/datasets/ds_15_mortgage_presence.csv',
    'https://code.s3.yandex.net/datasets/ds_15_credit_rating.csv',
    'https://code.s3.yandex.net/datasets/ds_15_macro_data.csv',
    'https://code.s3.yandex.net/datasets/ds_15_cohort_grid.csv',
]

In [ ]:
def show_info(df) -> None:
    """отображение базовой информации о датасете"""

    display(df.info(), df.head(5))

In [ ]:
df_list = [
    pd.read_csv(path) for path in path_s
]

In [ ]:
for df in df_list:
    show_info(df) 

In [ ]:
df_names = [
    'loan_payment_credit',
    'transactions',
    'client_description',
    'credit_description',
    'mortgage_presence',
    'credit_rating',
    'macro_data',
    'cohort_grid',
]

Итог:
- загрузка данных прошла корректно: типы данных соответствуют описанию

Анализ:
- данные содержат временные ряды. При дальнейшем анализе важно сохранять сортировку по датам 
- для макроэкономических показателей объединение будет проходить по дате. Для остальных таблиц объединение возможно по ID клиентов. 
- явных пропусков в данных нет


---
## 2. Исследовательский анализ данных


In [ ]:
# определение категориальных и числовых типов
CAT_COLS_TYPES = ['object', 'category']
NUM_COLS_TYPES = [np.number]

In [ ]:
def get_cat_num_cols(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    """Получить списки названий категориальных и числовых столбцов"""
    cat_cols = df.select_dtypes(include=CAT_COLS_TYPES).columns.tolist()
    num_cols = df.select_dtypes(include=NUM_COLS_TYPES).columns.tolist()

    return cat_cols, num_cols

def get_base_df_info(name: str, df: pd.DataFrame):
    """Информация о датафрейме для ИАД анализа"""
    
    rows_count, cols_count = df.shape
    missing = df.isna().sum()
    missing_share = (missing / rows_count * 100).round(2) if rows_count else pd.Series(0, index=df.columns, dtype=float)
    duplicates_count = (df.duplicated().sum() / rows_count * 100).round(2)
    unique_values = df.nunique(dropna=False)

    cat_cols, num_cols = get_cat_num_cols(df=df)

    overview_df = pd.DataFrame({
        'Метрика': ['Строк', 'Колонок', 'Явных дубликатов, %', 'Категориальных колонок', 'Числовых колонок'],
        'Значение': [rows_count, cols_count, duplicates_count, int(len(cat_cols)), int(len(num_cols))],
    })

    summary_df = pd.DataFrame({
        'Тип данных': df.dtypes.astype(str),
        'Пропуски': missing,
        'Доля пропусков, %': missing_share,
        'Уникальные значения': unique_values,
    }).sort_values(by=['Пропуски', 'Уникальные значения'], ascending=[False, False])

    cols_df = pd.DataFrame({
        'Категориальные колонки': pd.Series(cat_cols),
        'Числовые колонки': pd.Series(num_cols),
    })

    print(f'Общая информация о датафрейме {name}')
    display(df.head())
    display(overview_df)

    print('Сводка по признакам')
    display(summary_df)

    print('Типы колонок')
    display(cols_df)
    print('-'*50)

### Анализ таблиц 

- размеры, типы данных, пропуски, дубли

In [ ]:
for name, data_frame in zip(df_names, df_list):
    get_base_df_info(name=name, df=data_frame)

Анализ:
- в таблице по наличию ипотеки все 6609 значения константные и равны 1. При объединении будем предполагать, что у остальных этот признак равен 0 (то есть ипотеки нет)
- данные не содержат дублей и пропусков

### Анализ распределений 

#### Числовые данные

In [ ]:
def plot_hist_box(
    df: pd.DataFrame, feature_name: str,
    df_name: str,
    x_label: str = 'значение',
):
    fig, axes = plt.subplots(nrows=2, ncols=1)
    axes[0].grid(True)
    axes[1].grid(True)
    sns.histplot(data=df, x=feature_name, stat='density', ax=axes[0])
    sns.boxplot(data=df, x=feature_name, orient="h", ax=axes[1], label=f"{df[feature_name].describe()}")
    axes[0].set_title(f'Распределение признака {feature_name} в датасете {df_name}')
    axes[0].set_ylabel('плотность')
    axes[1].set_xlabel(x_label)
    axes[1].set_ylabel(feature_name)
    axes[1].legend(fontsize="x-small", loc='upper right')
    plt.show()

In [ ]:
for df, df_name in zip(df_list, df_names):
    _, num_col_names = get_cat_num_cols(df)
    for name in num_col_names:
        plot_hist_box(df=df, feature_name=name, df_name=df_name)

Анализ:
- значение дней просрочек равномерно распределено по диапазону
- траты по кодам MCC имеют скошенное влево распределение. Присутствуют малочисленные выбросы -- большие траты
- возрастные группы представлены равномерно 
- наличие иждевенцев представлено в примерно равных долях 
- по доходу распределение равномерное 
- по сумме кредита распределение равномерное
- данные по наличию ипотеки принимают одно значнение: 1
- распределение по кредитному рейтингу симметричное, присутсвуют выбросы по краям. Имеются сгустки плотности на некоторых значениях. Возможно, это результат округления или стандратные крейтинги для каких-то категорий граждан
- Распределение по учетной ставке несимметричное, присутствуют малочисленные выбросы (высокая ставка -- достаточно редкое явление)
- По безработице распределение симметричное без выбросов
- По инфляции распределение симметричное, есть выбросы на правом краю диапазона значений

#### Категориальные данные

In [ ]:
def plot_count_bars(
    df: pd.DataFrame, col_name: str, df_name: str
):
    """Столбчатые диаграммы распрелелений категориальных признаков"""
    df = df.sort_values(by=name) # сортировка по значениям
    sns.countplot(data=df, x=col_name)
    plt.title(f'Распределение признака {col_name} в датасете {df_name}')
    plt.ylabel('количество')
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
for df, df_name in zip(df_list, df_names):
    cat_col_names, _ = get_cat_num_cols(df)
    cat_col_names = [name for name in cat_col_names if name not in ['ID']]
    for name in cat_col_names:
        plot_count_bars(df=df, col_name=name, df_name=df_name)

Анализ:
- неявных пропусков в виде флагов невалидности нет
- дата начала периода просрочки: -- линейный рост количества записей до середины 2014 года, долее распределение равномерное, со 2ой половины 2018 количество записей в месяц существенно возросло (в 2 раза)
- Количество записей о датах транзакций растет линейно 
- по признаку семейного пложения дисбаланса нет 
- даты регистраций распределены равномерно 
- по датам открытия ипотеки распределение не имеет четкой структуры: в основном распрелеление равномерное, но местами есть пики и просадки
- количество дат датасете по кредитным рейтингам растет линейно 
- даты записей макроэкономических показателей равномерно распрелены - по одной записи на дату
- количество дат скоринга растет линейно

### Вывод по ИАД

#### Общие сведения

Данные представлены в виде 8 таблиц:
- Данные о просрочке платежа: loan_payment_credit
- Месячные транзакции клиента: transactions
- Описание клиента на момент регистрации в банке: client_description
- Описание кредита: credit_description
- Данные о наличии ипотеки: mortgage_presence
- Данные о кредитном рейтинге клиента: credit_rating
- Данные о макроэкономических показателях России: macro_data
- Данные о дате проведения поведенческого скоринга: cohort_grid

Типы данных соотвествуют описанию. 

#### Пропуски, дубли

- явных и неявных пропусков/дублей в данных не обнаружено


#### Анализ распределений

Количественные данные:
- loan_payment_credit: значение дней просрочек равномерно распределено по диапазону
- transactions: траты по кодам MCC имеют скошенное влево распределение. Присутствуют малочисленные выбросы -- большие траты
- client_description: по возрасту распределение равномерное
- client_description: наличие иждевенцев представлено в примерно равных долях 
- credit_description: по доходу и сумме кредита распределение равномерное 
- mortgage_presence: данные по наличию ипотеки принимают одно значнение - 1
- credit_rating: распределение по кредитному рейтингу симметричное, присутсвуют выбросы по краям. Имеются сгустки плотности на некоторых значениях. Выдвинуто предположение, что в происходило округление или имеются стандратные рейтинги для каких-то категорий граждан.
- macro_data: Распределение по учетной ставке несимметричное и имеет малый набор уникальных значений
- macro_data: По безработице распределение симметричное без выбросов
- macro_data: По инфляции распределение симметричное, есть выбросы 

Категориальные данные:
- loan_payment_credit: дата начала периода просрочки: -- линейный рост количества записей до середины 2014 года, долее распределение равномерное, со 2ой половины 2018 количество записей в месяц существенно возросло (в 2 раза)
- transactions: Количество записей о датах транзакций растет линейно 
- client_description: по признаку семейного пложения дисбаланса нет 
- credit_rating: даты регистраций распределены равномерно 
- mortgage_presence: по датам открытия ипотеки распределение не имеет четкой структуры: в основном распрелеление равномерное, но местами есть пики и просадки
- credit_rating: количество дат в датасете по кредитным рейтингам растет линейно 
- macro_data: даты записей макроэкономических показателей равномерно распределены - по одной записи на дату
- cohort_grid: количество дат скоринга растет линейно



---
## 3. Формирование датасета

### Формирование целевой переменной

1. Значение бинарной целевой переменной нужно определить для каждой строки со столбцами `ID` и `score_date` в таблице `cohort_grid`.

2. Таргет равен 1 при соблюдении двух условий:
    * Если значение в поле `просрочка_дней` больше или равно 90.
    * Если для клиента существует строка в таблице `loan_payment_credit`, где значение в поле `дата_начала_периода` попадает в интервал `[score_date, score_date + 365 дней)`.

>Важно: у клиента может быть несколько эпизодов с просрочками от 90 дней. Вам нужно взять первый по времени возникновения эпизод в таблице с просрочками.

3. После расчёта целевой переменной удалите строки, где дефолт уже произошёл к моменту скоринга, то есть `дата_начала_периода < score_date`. Это необходимо, так как для корректной работы с временной структурой важно учитывать дефолты, произошедшие в прошлом относительно даты скоринга.

In [ ]:
cohort_grid = df_list[7].copy()
cohort_grid['score_date'] = pd.to_datetime(cohort_grid['score_date'])
loan_payment_credit = df_list[0].rename(columns={'дата_начала_периода': 'начало_периода_просрочки'})
loan_payment_credit['начало_периода_просрочки'] = pd.to_datetime(loan_payment_credit['начало_периода_просрочки'])

In [ ]:
def create_target(loan_payment_credit, cohort_grid):
    # первый эпизод просрочки с продолжительностью более 90 дней
    first_default = (
        loan_payment_credit.loc[loan_payment_credit['просрочка_дней'].ge(90)]
        .groupby('ID', as_index=False)['начало_периода_просрочки']
        .min()
        .rename(columns={'начало_периода_просрочки': 'first_default_date'})
    )
    # совмещаем данные скоринга и данные о первых дефолтах
    df_result = cohort_grid.merge(first_default, on='ID', how='left', validate='many_to_one')

    # смещение на 12 месяцев
    forecast_horizon = df_result['score_date'] + pd.DateOffset(months=12) 

    # целевой класс
    df_result['target'] = (
        df_result['first_default_date'].gt(df_result['score_date'])
        & df_result['first_default_date'].le(forecast_horizon)
    ).astype('int8')

    # исключение строк, где дефолт произошел до даты скоринга
    df_result = df_result[
        df_result['first_default_date'].isna() # не было дефолтов
        | df_result['first_default_date'].ge(df_result['score_date'])
    ].copy()

    df_result = df_result.drop(columns='first_default_date')

    return df_result

In [ ]:
df_result = create_target(loan_payment_credit, cohort_grid)
df_result

### Создание итоговой таблицы

In [ ]:
transactions = df_list[1].copy()
transactions['date'] = pd.to_datetime(transactions['date'])
transactions = transactions.rename(columns={'date': 'date_transactions'})

In [ ]:
# столбцы транзакции для агрегации по прошедшим месяцам
mcc_cols = [
    'MCC_5300', 'MCC_5814', 'MCC_5812', 'MCC_5411',
    'MCC_3990', 'MCC_5722', 'MCC_4900', 'MCC_другое'
]

transactions = (
    transactions
    .sort_values(['ID', 'date_transactions'])
    .reset_index(drop=True)
)

In [ ]:
# проверка, что пары уникальны
transactions.duplicated(['ID', 'date_transactions']).any()

In [ ]:
# Исключаем текущий месяц: используем только значения до score_date
past_transactions = transactions.groupby('ID')[mcc_cols].shift(1)

features = transactions[['ID', 'date_transactions']].copy()

for window in [3, 6]: # агрегация за периоды
    rolling_sum = (
        past_transactions
        .groupby(transactions['ID'])
        .rolling(window=window, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
        .sort_index()
    )

    rolling_sum.columns = [
        f'{column}_sum_{window}m' for column in mcc_cols
    ]

    features = features.join(rolling_sum)

features = features.rename(columns={'date_transactions': 'score_date'})

In [ ]:
rows_before = len(df_result)

df_result = df_result.merge(
    features,
    on=['ID', 'score_date'],
    how='left',
    validate='one_to_one'
)



In [ ]:
df_result

In [ ]:
client_description = df_list[2].copy()
client_description['дата_регистрации'] = pd.to_datetime(client_description['дата_регистрации'])
client_description

In [ ]:
# ближайшая запись в прошлом с информацией о клиенте
df_result = pd.merge_asof(
    df_result.sort_values('score_date'),
    client_description.sort_values('дата_регистрации'),
    by='ID',
    left_on='score_date',
    right_on='дата_регистрации',
    direction='backward',
    allow_exact_matches=False,
)
df_result = df_result.drop(columns='дата_регистрации')

In [ ]:
credit_description = df_list[3].copy()
credit_description

In [ ]:
df_result = df_result.merge(credit_description, on='ID', how='left', validate='many_to_one')

In [ ]:
df_result

In [ ]:
mortgage_presence = df_list[4].copy()
mortgage_presence

In [ ]:
df_result = df_result.merge(mortgage_presence, on='ID', how='left', validate='many_to_one')
df_result['наличие_ипотеки'] = (
    df_result['дата_открытия'].notna()
    & df_result['дата_открытия'].lt(df_result['score_date'])
).astype('int8')
df_result = df_result.drop(columns='дата_открытия')
df_result

In [ ]:
credit_rating = df_list[5].copy()
credit_rating = credit_rating.rename(columns={'date': 'date_credit_rating'})
credit_rating['date_credit_rating'] = pd.to_datetime(
    credit_rating['date_credit_rating']
)
credit_rating

In [ ]:
# ближейший кредитный рейтинг в прошлом для заданной даты
df_result = pd.merge_asof(
    df_result.sort_values('score_date'),
    credit_rating.sort_values('date_credit_rating'),
    by='ID',
    left_on='score_date',
    right_on='date_credit_rating',
    direction='backward',
    allow_exact_matches=False,
)
df_result = df_result.drop(columns='date_credit_rating')

In [ ]:
# макро экономические показатели за предыдущий известный месяц
macro_data = df_list[6].copy()
macro_data = macro_data.rename(columns={'date': 'macro_date'})
macro_data['macro_date'] = pd.to_datetime(macro_data['macro_date'])

df_result = pd.merge_asof(
    df_result.sort_values('score_date'),
    macro_data.sort_values('macro_date'),
    left_on='score_date',
    right_on='macro_date',
    direction='backward',
    allow_exact_matches=False
)
df_result = df_result.drop(columns='macro_date')


In [ ]:
df_result.columns

### Создание новых признаков

1. Признак:
- **income_diff = доход_за_3_месяца  / суммарные_траты_по_всем_категориям_за_3_месяца**

Предположение:
- если в течение какого-то срока траты клиента превосходят его доход, то в будущем возрастает вероятность неплатежа по задолженности 

In [ ]:
# отношение суммарного дохода к тратам за 3 месяца
mcc_3m_cols = [
    'MCC_5300_sum_3m', 'MCC_5814_sum_3m',
    'MCC_5812_sum_3m', 'MCC_5411_sum_3m',
    'MCC_3990_sum_3m', 'MCC_5722_sum_3m',
    'MCC_4900_sum_3m', 'MCC_другое_sum_3m',
]

spend_3m = df_result[mcc_3m_cols].sum(axis=1, min_count=1)

df_result['income_spend_ratio_3m'] = (
    3 * df_result['доход'] / spend_3m.replace(0, np.nan)
)

In [ ]:
df_result.columns

2. Признак:
- **prev_default_counts** -- количество дефолтов за прошедшие 12 месяцев

Предположение:
- высокое количество дефолтов в прошлом может говорить о том, что высока веротяность просрочки и в будущем 

In [ ]:
df_result['prev_default_counts'] = (
    df_result
    .sort_values(by='score_date')
    .groupby('ID')['target']
    .shift(1)
    .rolling(window=12, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
    .sort_index()
)
df_result

### Анализ итоговой таблицы

Проверка целевой переменой

In [ ]:
df_result['target'].sum() / len(df_result['target']) * 100

Анализ:
- наблюдается дисбаланс классов в целевой переменной

Рекомендация:
- необходимо воспользоваться оверсэмплингом/андерсэмплингом в пайплайне предобработки данных для улучшения качества предсказания

In [ ]:
# контроль отсутствия дублей
df_result.duplicated(['ID', 'score_date']).any()

In [ ]:
# колонки с пропусками 
len(df_result.columns[df_result.isna().any()].to_list())

#### Промежуточный итог по созданию общей таблицы

- получено большое количество признаков (более 30). Это может вызвать трудности при обучении, поскольку вычислительные затраты на обработку такого массива данных значительные. Для решения этой проблемы необходимо отобрать на валидации наиболее информативные признаки
- в рамках целевой переменной есть дисбаланс. В пайплайн обучения необходимо встроить методы балансировки 
- большинство временных рядов содержат пропуски. Их необходимо обработать 

---
## 4. Пайплайн предобработки данных

In [ ]:
df_result.info()

In [ ]:
get_cat_num_cols(df=df_result)

In [ ]:
# числовые признаки, которые по смыслу явяются категориальными
NUM_SPECIAL_MOST_FREQ = ['наличие_иждивенцев'] # замена пропусков на наиболее часто встречающееся значение
NUM_SPECIAL_ZERO = ['наличие_ипотеки']  # замена на флаг отсутсвия - 0

In [ ]:
# отбор признаков

def select_cat_one_hot(X):
    cat_cols = X.select_dtypes(include=CAT_COLS_TYPES).columns.tolist()
    return [name for name in cat_cols if X[name].nunique() < 10]

def select_cat_ordinal(X):
    cat_cols = X.select_dtypes(include=CAT_COLS_TYPES).columns.tolist()
    return [name for name in cat_cols if X[name].nunique() > 10]

def select_num_classic(X):
    num_cols_continues = X.select_dtypes(include=NUM_COLS_TYPES).columns.tolist()
    special_num_cats = NUM_SPECIAL_ZERO + NUM_SPECIAL_MOST_FREQ
    return [name for name in num_cols_continues if name not in special_num_cats]

def select_num_special_zero(X):
    num_cols = X.select_dtypes(include=NUM_SPECIAL_ZERO).columns.tolist()
    return num_cols

def select_num_special_mf(X):
    num_cols = X.select_dtypes(include=NUM_SPECIAL_MOST_FREQ).columns.tolist()
    return num_cols

In [ ]:
class IQROutliersReplacer(BaseEstimator, TransformerMixin):
    """Фильтрация выбросов по межквартильному размаху с заданным коэффициентом"""
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bound_ = None
        self.upper_bound_ = None

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.lower_bound_ = X.quantile(0.25, axis=0) - self.factor * X.quantile(0.75, axis=0) - self.lower_bound_
        self.upper_bound_ = X.quantile(0.75, axis=0) + self.factor * X.quantile(0.75, axis=0) - self.upper_bound_
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        for col in X.columns:
            X.loc[(X[col] < self.lower_bound_[col]) | (X[col] > self.upper_bound_[col]), col] = np.nan
        return X


In [ ]:
def create_preprocessing_pipeline(
        with_outliers_filtering: bool = False
):
    """
    Создание пайплайна предобратботки данных.

    Args:
        with_outliers_filtering (bool): флаг наличия фильтации выбросов

    Returns:
        Pipeline: Пайплайн предобработки данных
    """
    
    # категориальные признаки

    # малочисленные
    cat_one_hot_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
    ])
    # многочисленные
    cat_ordinal_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder())
    ])

    # количественные данные, которые по смыслу являются категориальными
    # замена пропусков на наиболее часто встречающееся значение
    num_special_mf_pipeline = Pipeline(steps=[
        ('scaler', MinMaxScaler()),
        ('imputer', SimpleImputer(strategy='most_frequent')),
    ])

    # особые признаки: отсутстие значения = 0
    num_special_zero_pipeline = Pipeline(steps=[
        ('scaler', MinMaxScaler()),
        ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ])

    # количественные признаки с возможными выбросами
    num_classic_steps = []
    if with_outliers_filtering:
        num_classic_steps.append(
            ('outliears', IQROutliersReplacer())
        )
    num_classic_steps += [
        ('scaler', RobustScaler()),
        ('imputer', SimpleImputer(strategy='median'))
    ]

    num_classic_pipeline = Pipeline(steps=num_classic_steps)

    transformers = [
        ('cat_one_hot_pipeline', cat_one_hot_pipeline, select_cat_one_hot),
        ('cat_ordinal_pipeline', cat_ordinal_pipeline, select_cat_ordinal),
        ('num_special_mf_pipeline', num_special_mf_pipeline, select_num_special_mf),
        ('num_special_zero_pipeline', num_special_zero_pipeline, select_num_special_zero),
        ('num_classic_pipeline', num_classic_pipeline, select_num_classic),
    ]
    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder='drop'
    )


    result_pipeline = Pipeline([
        ('prep', preprocessor)
    ])

    return result_pipeline


--- 
## 5. Базовые модели


### Базовые модели

1. Подготовьте обучающую, калибровочную и тестовую выборки. Разбейте обучающую на три фолда для последующего использования кросс-валидации. Для оценки качества и калибровки используйте размер выборки, равный 12 месяцам.


2. При необходимости проведите категоризацию данных, применив нужный Encoder и использовав пайплайн.

3. Обучите базовые модели с кросс-валидацией по трём фолдам:
    * Две базовые модели — логистическую регрессию и случайный лес — без балансировки классов в целевой переменной.
    * Логистическую регрессию и случайный лес с балансировкой классов. Выберите метод балансировки самостоятельно. Обязательно примените хотя бы один метод. Можно попробовать несколько и выбрать лучший.
    * Сделайте выводы о работе всех четырёх моделей.

4. Случайный лес с настройками по умолчанию легко переобучается, потому что запоминает обучающую выборку, из-за чего модель может терять в качестве на новых данных. Логистическая регрессия же сразу готова к работе за счёт встроенной L2-регуляризации, которая автоматически контролирует сложность модели.

   Чтобы исправить проблемы модели Random Forest, вам нужно подобрать для неё гиперпараметры с помощью  Optuna. Количество гиперпараметров должно быть не менее трёх. Для оптимизации используйте метрику missed defaults rate.

5. Сравните все полученные модели.

6. Для оценки моделей используйте метрики:
   * accuracy или ROC-AUC,
   * approval rate,
   * default rate,
   * missed defaults rate.

7. Сделайте вывод о работе, проделанной в этом разделе.

#### Разбиение выборки на train/validate/test

In [ ]:
# преобразование временных меток в индексы
df_result.set_index('score_date', inplace=True) 
df_result.sort_index(inplace=True)


In [ ]:
df_result.head()

In [ ]:
df_result = df_result.reset_index().drop(columns='score_date')

In [ ]:
# разделение на обучающую и отложенную выборки
split_point = int(len(df_result) * 0.8)
train_data = df_result.iloc[:split_point]
test_data = df_result.iloc[split_point:]

In [ ]:
X_train = train_data.drop(columns='target')
y_train = train_data['target']
X_train.shape, y_train.shape

In [ ]:
X_test = test_data.drop(columns='target')
y_test = test_data['target']
X_test.shape, y_test.shape

In [ ]:
# --- CV-параметры ---
tscv = TimeSeriesSplit(n_splits=3)

#### Базовые модели для обучения

In [ ]:
# логрег
logreg_base_pipeline = Pipeline(steps=[
    ('preproc', create_preprocessing_pipeline()),
    ('model', LogisticRegression(random_state=RANDOM_SEED))
])

In [ ]:
# случайный лес
random_forest_base_pipeline = Pipeline(steps=[
    ('preproc', create_preprocessing_pipeline()),
    ('model', RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, max_depth=100))
])

#### Обучение кросс валидацией и оценка качества

In [ ]:
def calculate_approval_metrics(y_true, y_pred):
    """
    Рассчитывает Approval Rate, Default Rate, Missed Defaults Rate
    для заданного порога.
    
    Args:
        y_true : array-like, истинные метки (1 - дефолт, 0 - хороший)
        y_pred_proba : array-like, предсказанные вероятности дефолта
        threshold : float, порог отбора (одобряем если proba < threshold)
    
    Returns:
        dict с метриками
    """
    
    # Количество одобренных и общее число клиентов
    n_approved = y_pred.sum()
    n_total = len(y_true)
    
    # Фактические дефолты среди одобренных
    defaults_among_approved = np.sum(y_true[y_pred] == 1)
    
    # Общее количество дефолтов в выборке
    total_defaults = np.sum(y_true == 1)
    
    approval_rate = n_approved / n_total
    default_rate = defaults_among_approved / n_approved if n_approved > 0 else 0.0
    missed_defaults_rate = defaults_among_approved / total_defaults if total_defaults > 0 else 0.0
    
    return {
        'approval_rate': approval_rate,
        'default_rate': default_rate,
        'missed_defaults_rate': missed_defaults_rate,
    }

In [ ]:
def cross_valid_cycle(model, tscv, X: pd.DataFrame, y: pd.Series):
    scores_list = []

    for fold, (train_index, valid_index) in enumerate(tscv.split(X)):
        
        # 2.1. Разделение данных для текущего фолда
        X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
        y_train, y_valid = y[train_index], y[valid_index]
        train_dates = df.index[train_index]
        valid_dates = df.index[valid_index]

        # 2.2. Обучение модели: использование RandomForestRegressor
        # n_estimators=100 - количество деревьев, random_state для воспроизводимости
        model.fit(X_train, y_train)

        # 2.3. Прогнозирование
        y_pred = model.predict(X_valid)

        # 2.4. Оценка качества
        scores = calculate_approval_metrics(y_true=y_valid, y_pred=y_pred)

    return scores_list

In [ ]:
def plot_score_by_folds(scores_list: list[dict]):
    """Визуализация метрик по фолдам кросс валидации"""
    folds_n = []
    approval_rates = []
    default_rates = []
    missed_defaults_rates = []

    for idx, metrics in enumerate(scores_list):
        folds_n.append(idx)
        approval_rates.append(metrics['approval_rate'])
        default_rates.append(metrics['default_rate'])
        missed_defaults_rates.append(metrics['missed_defaults_rate'])

    fig, axes = plt.subplots(len(scores_list), 1)

    metrics = [approval_rates, default_rates, missed_defaults_rates]

    for ax, metric in zip(axes, metrics):
        ax.plot(folds_n, metric)

        ax.set_ylabel('Значение')
        ax.legend()
        ax.grid(True, linestyle=':', alpha=0.5)
        
    plt.show()

In [ ]:
logreg_base_result = cross_valid_cycle(
    model=logreg_base_pipeline,
    tscv=tscv,
    X=X_train,
    y=y_train
)


## Калибровка модели и пересчёт результатов

* Проведите калибровку лучшей версии модели. Используйте отдельную калибровочную выборку.
* Используйте метод, подходящий для случайного леса.
* Постройте график калибровки.
* Сделайте вывод, оцените результаты с помощью коэффициента Бриера.

## Поиск порога решения

* Используя откалиброванную модель и калибровочную выборку, найдите порог, при котором будут достигнуты заданные в постановке задачи значения метрик:
    * approval rate — не менее 65%;
    * default rate — не более 2%;
    * missed defaults rate — не более 4%.
    
* Сделайте вывод о достигнутых в этом разделе результатах.

## Анализ матрицы ошибок

* Оцените стабильность модели на тестовых данных. Для этого постройте:
    * матрицу ошибок на калибровочных данных;
    * матрицу классификации на тестовых данных.
* Сделайте вывод о моделях, рассчитав классические метрики машинного обучения и указанные в ТЗ бизнес-метрики.
* Сделайте вывод о стабильности модели.

## Фиксирование итоговой модели

- Опишите лучшую модель и найденный порог классификации.


## Анализ важности признаков

* Проведите анализ важности признаков найденной модели на полных тренировочных данных.
* Используйте `feature_importances_` для найденной модели.
* Сделайте вывод о силе влияния признаков на дефолт.

## Выводы по проекту

Сделайте выводы по проекту. Можете использовать такой план:

1. Цель и задачи исследования.

2. Подготовка данных и выборок.

3. Поиск и настройка модели.

4. Калибровка вероятностей.

5. Оптимизация бизнес-порога.

6. Анализ важности признаков.

7. Финальный пайплайн.

8. Основные выводы и рекомендации для бизнеса.
